# Problem 44: Wild Mushroom Toxicity Identification & Poison Prevention
Decision Tree (CART/Gini), Cost-Complexity Pruning, Bernoulli Naive Bayes.
**Google Colab Link:** *(paste your shared URL here and on page 1 of the report)*

Convention: **poisonous = positive class (1)**, so a *false negative* = poisonous mushroom called edible (lethal).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, recall_score
SEED=42; np.random.seed(SEED)
plt.rcParams.update({"figure.dpi":120,"axes.grid":True,"grid.alpha":.3})

## Phase 1: Data Exploration

In [ ]:
cols=["class","cap-shape","cap-surface","cap-color","bruises","odor","gill-attachment","gill-spacing","gill-size",
"gill-color","stalk-shape","stalk-root","stalk-surface-above-ring","stalk-surface-below-ring","stalk-color-above-ring",
"stalk-color-below-ring","veil-type","veil-color","ring-number","ring-type","spore-print-color","population","habitat"]
URL="https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data"
try:
    df=pd.read_csv(URL,header=None,names=cols)
except Exception:
    from ucimlrepo import fetch_ucirepo   # pip install ucimlrepo
    m=fetch_ucirepo(id=73); df=pd.concat([m.data.targets,m.data.features],axis=1)
    df.columns=cols if len(df.columns)==len(cols) else df.columns
print(df.shape)
df.head()

In [ ]:
print("Class balance:\n",df["class"].value_counts().rename({"e":"edible","p":"poisonous"}))
print("\nMissing ('?') per column:\n",(df=="?").sum()[lambda s:s>0])
audit=pd.DataFrame({"n_levels":df.nunique(),"levels":[sorted(df[c].unique()) for c in df]})
audit   # note: veil-type has ONE level (zero-information constant)

In [ ]:
df["class"].value_counts().plot.bar(color=["tab:green","tab:red"]); plt.title("Class balance"); plt.xticks(rotation=0); plt.show()

## Phase 2: One-Hot Encoding
'?' in stalk-root is treated as its own category (`stalk-root_?`), not imputed: missingness itself is informative here.

In [ ]:
y=(df["class"]=="p").astype(int)
X=pd.get_dummies(df.drop(columns="class"),dtype=np.uint8)
print("Encoded shape:",X.shape," | sparsity (share of zeros): %.3f"%(1-X.values.mean()))
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=SEED)
print(Xtr.shape,Xte.shape)

## Phase 3: Unconstrained Tree Induction

In [ ]:
full=DecisionTreeClassifier(criterion="gini",random_state=SEED).fit(Xtr,ytr)
print("Train acc:",accuracy_score(ytr,full.predict(Xtr)),"| Test acc:",accuracy_score(yte,full.predict(Xte)))
print("Leaves:",full.get_n_leaves(),"| Depth:",full.get_depth())
# Gini vs entropy
for crit in ["gini","entropy"]:
    t=DecisionTreeClassifier(criterion=crit,random_state=SEED).fit(Xtr,ytr)
    print(crit,"-> leaves",t.get_n_leaves(),"depth",t.get_depth(),"test acc",accuracy_score(yte,t.predict(Xte)))

## Phase 4: Pruning Optimisation

In [ ]:
path=full.cost_complexity_pruning_path(Xtr,ytr)
alphas=path.ccp_alphas[:-1]            # drop the alpha that collapses tree to the root
trees=[DecisionTreeClassifier(random_state=SEED,ccp_alpha=a).fit(Xtr,ytr) for a in alphas]
leaves=[t.get_n_leaves() for t in trees]; depth=[t.get_depth() for t in trees]
tr_acc=[t.score(Xtr,ytr) for t in trees]; te_acc=[t.score(Xte,yte) for t in trees]
skf=StratifiedKFold(5,shuffle=True,random_state=SEED)
cv_acc=[cross_val_score(DecisionTreeClassifier(random_state=SEED,ccp_alpha=a),Xtr,ytr,cv=skf).mean() for a in alphas]
res=pd.DataFrame({"alpha":alphas,"leaves":leaves,"depth":depth,"train_acc":tr_acc,"cv_acc":cv_acc,"test_acc":te_acc})
res.tail(12)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].step(alphas,leaves,where="post",label="leaves"); ax[0].step(alphas,depth,where="post",label="depth")
ax[0].set_xscale("symlog",linthresh=1e-4); ax[0].set_xlabel("ccp_alpha"); ax[0].set_ylabel("count"); ax[0].legend(); ax[0].set_title("Tree size vs ccp_alpha")
ax[1].plot(alphas,tr_acc,drawstyle="steps-post",label="train"); ax[1].plot(alphas,cv_acc,drawstyle="steps-post",label="5-fold CV")
ax[1].plot(alphas,te_acc,drawstyle="steps-post",label="test")
ax[1].set_xscale("symlog",linthresh=1e-4); ax[1].set_xlabel("effective alpha"); ax[1].set_ylabel("accuracy"); ax[1].legend(); ax[1].set_title("Accuracy vs ccp_alpha")
plt.tight_layout(); plt.savefig("fig_pruning.png",dpi=300); plt.show()

In [ ]:
# Select by CV (not test) to avoid leakage: largest alpha whose CV accuracy equals the best CV accuracy
best=res.cv_acc.max(); cand=res[res.cv_acc>=best-1e-12]
pick=cand.sort_values(["alpha"]).iloc[-1]; print(pick)
pruned=DecisionTreeClassifier(random_state=SEED,ccp_alpha=pick.alpha).fit(Xtr,ytr)
print("Test acc of selected subtree:",pruned.score(Xte,yte),"| leaves",pruned.get_n_leaves(),"| depth",pruned.get_depth())
# Test-set confirmation: simplest tree with 100% test accuracy
perfect=res[res.test_acc==1.0].sort_values(["leaves","depth"]).head(1); print("Simplest tree with 100% test acc:\n",perfect)
# Pre-pruning comparison
gs=GridSearchCV(DecisionTreeClassifier(random_state=SEED),{"max_depth":[2,3,4,5,6,None],"min_samples_split":[2,10,50],"min_samples_leaf":[1,5,20]},cv=skf).fit(Xtr,ytr)
print("Best pre-pruning:",gs.best_params_,"CV acc",round(gs.best_score_,4),"test acc",gs.score(Xte,yte))

## Mandatory visualisations

In [ ]:
plt.figure(figsize=(22,10))
plot_tree(pruned,feature_names=X.columns,class_names=["edible","poisonous"],filled=True,max_depth=3,fontsize=8,impurity=True,proportion=False)
plt.title("Pruned tree (top 4 levels): Gini impurity and sample distribution"); plt.savefig("fig_tree.png",dpi=300,bbox_inches="tight"); plt.show()

In [ ]:
imp=pd.Series(pruned.feature_importances_,index=X.columns)
imp_attr=imp.groupby(lambda c:c.rsplit("_",1)[0]).sum().sort_values()   # aggregate dummies back to the 22 attributes
imp_attr=imp_attr[imp_attr>0]
imp_attr.plot.barh(figsize=(8,5),color="tab:purple"); plt.xlabel("Gini importance (summed over one-hot levels)")
plt.title("Feature importance by attribute"); plt.tight_layout(); plt.savefig("fig_importance.png",dpi=300); plt.show()
imp_attr.sort_values(ascending=False).head(6)

In [ ]:
pred=pruned.predict(Xte); cm=confusion_matrix(yte,pred)
ConfusionMatrixDisplay(cm,display_labels=["edible","poisonous"]).plot(cmap="Blues",values_format="d"); plt.grid(False)
plt.title("Pruned tree: test confusion matrix"); plt.savefig("fig_cm.png",dpi=300); plt.show()
tn,fp,fn,tp=cm.ravel(); print("False negatives (poisonous->edible):",fn,"| False positives:",fp)
print(classification_report(yte,pred,target_names=["edible","poisonous"],digits=4))

## Phase 5: Critical Rule Extraction

In [ ]:
print(export_text(pruned,feature_names=list(X.columns),max_depth=5))

In [ ]:
# Odor as a single-handed rule: foul/fishy/creosote/spicy/musty/pungent (f,y,c,s,m,p) => poisonous
bad=df["odor"].isin(list("fyscmp")); poison=df["class"]=="p"
print("Odor-in-{foul,fishy,creosote,spicy,musty,pungent} rule:")
print(" poisonous covered: %d of %d (%.1f%%)"%((bad&poison).sum(),poison.sum(),100*(bad&poison).sum()/poison.sum()))
print(" edible wrongly flagged:",(bad&~poison).sum())
print("\nOdor=none residue (needs further rules):\n",pd.crosstab(df.loc[df.odor=="n","spore-print-color"],df.loc[df.odor=="n","class"]))

## Model comparison: Tree vs Bernoulli Naive Bayes (and risk-averse thresholding)

In [ ]:
nb=BernoulliNB(alpha=1.0).fit(Xtr,ytr)
rows=[]
for name,m in [("Unpruned tree",full),("Pruned tree (CV-selected)",pruned),("Bernoulli NB",nb)]:
    p=m.predict(Xte); c=confusion_matrix(yte,p).ravel()
    rows.append({"model":name,"test_acc":accuracy_score(yte,p),"recall_poisonous":recall_score(yte,p),
                 "FN":c[2],"FP":c[1],"5fold_CV_acc":cross_val_score(m.__class__(**m.get_params()),Xtr,ytr,cv=skf).mean()})
comp=pd.DataFrame(rows).round(4); comp

In [ ]:
# Asymmetric risk: lower the 'poisonous' decision threshold for NB (cost of FN >> cost of FP)
pp=nb.predict_proba(Xte)[:,1]
for th in [0.5,0.2,0.05,0.01]:
    p=(pp>=th).astype(int); c=confusion_matrix(yte,p).ravel(); print(f"threshold {th}: FN={c[2]} FP={c[1]}")
# Expected cost with FN cost = 100 x FP cost
for name,m in [("Pruned tree",pruned),("Bernoulli NB",nb)]:
    c=confusion_matrix(yte,m.predict(Xte)).ravel(); print(name,"weighted cost:",100*c[2]+c[1])

## Discussion prompts for Section F
* **Zero training error** arises because the 22 attributes separate the classes almost perfectly in this dataset; this is *not* evidence of overfitting by itself. Check CV/test curves.
* **Bias-variance:** extreme pruning raises bias (accuracy drops); mild pruning shrinks the tree at no accuracy cost.
* **Caveat:** records are *hypothetical* specimens drawn from a field-guide description, so 100% accuracy will not transfer to real foraging. Recommend never relying on a model alone; odor `n` (none) is the dangerous residue.
* Compare tree vs NB error types under the asymmetric cost.